# Assignment 3: Milestone I Natural Language Processing
## Task 2 and Task 3
#### Student Name: (Fill in)
#### Student ID: (Fill in)

Environment: Python 3 and Jupyter notebook

Libraries used:
- pandas, numpy
- scikit-learn (TfidfVectorizer, LogisticRegression)
- gensim (FastText)
- scipy
- pathlib, re, collections

## Introduction
This notebook implements Task 2 (feature representation) and Task 3 (classification) for cosmetics and beauty reviews.

Task 2 outputs generated by this notebook:
- `count_vectors.txt` (sparse unigram count vectors using `vocab.txt`)
- `unweighted_vectors.txt` (unweighted average FastText word vectors)
- `weighted_vectors.txt` (TF-IDF weighted average FastText word vectors)

Task 3 experiments use a simple model first (Logistic Regression) with 5-fold cross-validation to compare feature sets and answer both required questions.

## Importing Libraries

In [1]:
# !pip install -q --upgrade pip setuptools wheel
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
import os
import gensim.downloader as gensim_api
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Task 2. Generating Feature Representations for Cosmetics/Beauty Reviews

### 2.1 Count Vector Representation (Bag-of-Words)

The count vector representation encodes each review as a sparse vector where each dimension corresponds to a vocabulary word (from `vocab.txt`) and the value is the raw term frequency of that word in the `review_text` (excluding `review_title`).

- Vocabulary is loaded from `vocab.txt` (word → integer index, alphabetically sorted).
- For each review, tokenise the pre-processed `review_text` (space-separated tokens produced by Task 1) and count occurrences of vocabulary words.
- Only non-zero counts are stored (sparse format).
- Output format per line: `#review_index,word_integer_index:word_freq,...`
- Saved to `count_vectors.txt`.

In [3]:
PROCESSED_CSV = "processed.csv"
VOCAB = "vocab.txt"
COUNT_VECTOR = "count_vectors.txt"


def GenerateCountVector():
    # Load vocabulary (word -> integer index)
    vocab = {}
    with open(VOCAB, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            word, idx = line.rsplit(":", 1)
            vocab[word] = int(idx)

    # Load processed reviews
    df = pd.read_csv(PROCESSED_CSV)
    # review_text contains space-separated tokens produced by Task 1
    review_texts = df["review_text"].fillna("").astype(str).tolist()

    # Build and write sparse count vectors
    with open(COUNT_VECTOR, "w", encoding="utf-8") as out:
        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Count only tokens that exist in the vocabulary
            counts = Counter(token for token in tokens if token in vocab)
            # Sort by word integer index for a consistent ordering
            sparse_entries = sorted(
                (vocab[word], freq) for word, freq in counts.items()
            )
            sparse_str = ",".join(f"{idx}:{freq}" for idx, freq in sparse_entries)
            out.write(f"#{review_idx},{sparse_str}\n")

    print(f"Count vectors saved to '{COUNT_VECTOR}' ({len(review_texts)} reviews).")


# Run
GenerateCountVector()

Count vectors saved to 'count_vectors.txt' (61284 reviews).


### 2.2 Word Embedding Vectors (FastText — Unweighted & Weighted)

A pretrained **FastText** model (`fasttext-wiki-news-subwords-300`, 300-dimensional) is loaded via `gensim.downloader` and used to build two document-level vector representations for each review (`review_text` only):

- **Unweighted** — simple arithmetic mean of the FastText vectors of all valid tokens in the review.
- **Weighted** — TF-IDF weighted mean: each token's vector is scaled by its TF-IDF score (fitted across the full corpus) before averaging, giving more weight to informative terms.

Reviews with no recognisable tokens receive a zero vector. Both representations are saved in the same sparse-style line format:

```
#<review_index>,<v0>,<v1>,...,<v299>
```

Outputs: `unweighted_vectors.txt`, `weighted_vectors.txt`.

In [4]:
UNWEIGHTED_VECTOR = "unweighted_vectors.txt"
WEIGHTED_VECTOR = "weighted_vectors.txt"
FASTTEXT_MODEL_NAME = "fasttext-wiki-news-subwords-300"


def GenerateEmbeddingVectors():
    # Skip regeneration when both output files already exist.
    if os.path.exists(UNWEIGHTED_VECTOR) and os.path.exists(WEIGHTED_VECTOR):
        print(f"Skip GenerateEmbeddingVectors: '{UNWEIGHTED_VECTOR}' and '{WEIGHTED_VECTOR}' already exist.")
        return

    # Load pretrained FastText model
    print(f"Loading FastText model '{FASTTEXT_MODEL_NAME}'")
    fasttext_model = gensim_api.load(FASTTEXT_MODEL_NAME)
    vector_size = fasttext_model.vector_size
    print(f"Model loaded. Vector size: {vector_size}")

    # Load processed reviews
    df = pd.read_csv(PROCESSED_CSV)
    review_texts = df["review_text"].fillna("").astype(str).tolist()
    print("Load review")
    # Fit TF-IDF over the full corpus (for weighted representation)
    # tokenizer=str.split preserves the already-cleaned tokens from Task 1
    tfidf = TfidfVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None)
    tfidf_matrix = tfidf.fit_transform(review_texts)
    tfidf_feature_names = tfidf.get_feature_names_out()
    tfidf_vocab = {word: idx for idx, word in enumerate(tfidf_feature_names)}

    # Generate and write vectors
    with open(UNWEIGHTED_VECTOR, "w", encoding="utf-8") as uw_out, open(
        WEIGHTED_VECTOR, "w", encoding="utf-8"
    ) as w_out:

        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Keep only tokens the FastText model knows
            valid_tokens = [t for t in tokens if t in fasttext_model]

            if valid_tokens:
                vectors = np.array([fasttext_model[t] for t in valid_tokens])

                # Unweighted: simple average of word vectors
                unweighted_vec = vectors.mean(axis=0)

                # Weighted: TF-IDF weighted average
                tfidf_row = tfidf_matrix[review_idx]
                weights = np.array(
                    [
                        tfidf_row[0, tfidf_vocab[t]] if t in tfidf_vocab else 0.0
                        for t in valid_tokens
                    ]
                )
                weight_sum = weights.sum()
                if weight_sum > 0:
                    weighted_vec = (vectors * weights[:, np.newaxis]).sum(
                        axis=0
                    ) / weight_sum
                else:
                    weighted_vec = unweighted_vec
            else:
                unweighted_vec = np.zeros(vector_size)
                weighted_vec = np.zeros(vector_size)

            uw_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in unweighted_vec) + "\n"
            )
            w_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in weighted_vec) + "\n"
            )

    print(f"Number of reviews: {len(review_texts)}")
    print(f"Unweighted vectors saved to '{UNWEIGHTED_VECTOR}'")
    print(f"Weighted vectors saved to '{WEIGHTED_VECTOR}'")


# Run
GenerateEmbeddingVectors()

Skip GenerateEmbeddingVectors: 'unweighted_vectors.txt' and 'weighted_vectors.txt' already exist.


## Task 3. Linear Regression on FastText Features

This section trains and evaluates a **Linear Regression** model to predict `review_rating` using the generated document vectors.

- Features compared:
  - Unweighted FastText vectors (`unweighted_vectors.txt`)
  - TF-IDF weighted FastText vectors (`weighted_vectors.txt`)
- Target: `review_rating`
- Metrics: MAE, MSE, RMSE, and R²

In [ ]:
def load_dense_vectors(file_path: str) -> np.ndarray:
    vectors = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            vec = [float(v) for v in parts[1:]]  # Skip #review_index
            vectors.append(vec)
    return np.array(vectors, dtype=np.float32)


# Load target
df = pd.read_csv(PROCESSED_CSV)
y_all = pd.to_numeric(df["review_rating"], errors="coerce")
valid_mask = y_all.notna().to_numpy()
y = y_all[valid_mask].to_numpy(dtype=np.float32)

# Load generated feature vectors
X_unweighted = load_dense_vectors(UNWEIGHTED_VECTOR)[valid_mask]
X_weighted = load_dense_vectors(WEIGHTED_VECTOR)[valid_mask]


def evaluate_linear_regression_5fold(X: np.ndarray, y: np.ndarray, name: str):
    """Evaluate Linear Regression using 5-fold cross-validation."""
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    print(f"\n{name} (5-Fold Cross-Validation Results)")
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    r2_scores = []

    fold_num = 1
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = float(np.sqrt(mse))
        r2 = r2_score(y_test, y_pred)

        mae_scores.append(mae)
        mse_scores.append(mse)
        rmse_scores.append(rmse)
        r2_scores.append(r2)

        print(
            f"  Fold {fold_num}: MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}"
        )
        fold_num += 1

    print(f"MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
    print(f"MSE  : {np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}")
    print(f"RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
    print(f"R²   : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")


print(f"Samples used: {len(y)}")
print(f"Feature size (unweighted): {X_unweighted.shape[1]}")
print(f"Feature size (weighted)  : {X_weighted.shape[1]}")

evaluate_linear_regression_5fold(
    X_unweighted, y, "Linear Regression with Unweighted FastText"
)
evaluate_linear_regression_5fold(
    X_weighted, y, "Linear Regression with Weighted FastText"
)

Samples used: 61283
Feature size (unweighted): 300
Feature size (weighted)  : 300
  Fold 1: MAE=0.6887, MSE=0.9034, RMSE=0.9504, R²=0.1924
  Fold 2: MAE=0.6928, MSE=0.9071, RMSE=0.9524, R²=0.1937
  Fold 3: MAE=0.6886, MSE=0.8907, RMSE=0.9438, R²=0.2038
  Fold 4: MAE=0.6915, MSE=0.9040, RMSE=0.9508, R²=0.2060
  Fold 5: MAE=0.6951, MSE=0.9140, RMSE=0.9560, R²=0.2010

Linear Regression with Unweighted FastText (5-Fold Cross-Validation Results)
MAE  : 0.6913 ± 0.0025
MSE  : 0.9038 ± 0.0076
RMSE : 0.9507 ± 0.0040
R²   : 0.1994 ± 0.0054
  Fold 1: MAE=0.6923, MSE=0.9127, RMSE=0.9554, R²=0.1840
  Fold 2: MAE=0.6975, MSE=0.9185, RMSE=0.9584, R²=0.1836
  Fold 3: MAE=0.6934, MSE=0.9027, RMSE=0.9501, R²=0.1931
  Fold 4: MAE=0.6964, MSE=0.9175, RMSE=0.9578, R²=0.1942
  Fold 5: MAE=0.6992, MSE=0.9254, RMSE=0.9620, R²=0.1910

Linear Regression with Weighted FastText (5-Fold Cross-Validation Results)
MAE  : 0.6958 ± 0.0026
MSE  : 0.9154 ± 0.0075
RMSE : 0.9567 ± 0.0039
R²   : 0.1892 ± 0.0045


## Task 3. Random Forest Classification (Configurable Features)

This experiment predicts `is_a_buyer` using **Random Forest** with:

- Unweighted FastText vectors
- TF-IDF weighted FastText vectors
- Optional structured features from Task 1:
  - `review_rating`
  - `price`
  - `avg_product_rating`
  - `product_rating_count`
  - `brand_encoded` (derived from `brand_name`)
  - `product_encoded` (derived from `product_title`)

You can control feature usage via configuration variables in the next code cell.

In [ ]:
USE_STRUCTURED_FEATURES = True
STRUCTURED_FEATURES = [
    # "review_rating",
    "price",
    # "avg_product_rating",
    "product_rating_count",
    "brand_encoded",
    "product_encoded",
]

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "class_weight": "balanced",
}


def load_dense_vectors(file_path: str) -> np.ndarray:
    vectors = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            vectors.append([float(v) for v in parts[1:]])  # skip #review_index
    return np.array(vectors, dtype=np.float32)


def build_structured_matrix(
    df_in: pd.DataFrame, feature_names: list[str]
) -> np.ndarray:
    cols = []

    for feat in feature_names:
        if feat == "brand_encoded":
            encoded = pd.factorize(df_in["brand_name"].fillna("unknown"))[0].astype(
                np.float32
            )
            cols.append(encoded.reshape(-1, 1))
        elif feat == "product_encoded":
            encoded = pd.factorize(df_in["product_title"].fillna("unknown"))[0].astype(
                np.float32
            )
            cols.append(encoded.reshape(-1, 1))
        else:
            if feat not in df_in.columns:
                raise ValueError(f"Feature '{feat}' not found in dataframe columns.")
            numeric_col = pd.to_numeric(df_in[feat], errors="coerce")
            filled_col = numeric_col.fillna(numeric_col.median()).to_numpy(
                dtype=np.float32
            )
            cols.append(filled_col.reshape(-1, 1))

    return np.hstack(cols) if cols else np.empty((len(df_in), 0), dtype=np.float32)


def prepare_binary_target(series: pd.Series) -> np.ndarray:
    if series.dtype == bool:
        return series.astype(int).to_numpy()

    numeric_try = pd.to_numeric(series, errors="coerce")
    if numeric_try.notna().all():
        return numeric_try.astype(int).to_numpy()

    str_series = series.astype(str).str.strip().str.lower()
    mapping = {"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0}
    mapped = str_series.map(mapping)

    if mapped.isna().any():
        bad_vals = sorted(str_series[mapped.isna()].unique().tolist())[:5]
        raise ValueError(
            f"Unable to map some label values in is_a_buyer, examples: {bad_vals}"
        )

    return mapped.astype(int).to_numpy()


def evaluate_rf_variant_5fold(
    name: str, X_embed: np.ndarray, y: np.ndarray, X_struct: np.ndarray | None
):
    print(f"\n{name} (5-Fold Stratified Cross-Validation Results)")
    """Evaluate Random Forest using 5-fold stratified cross-validation."""
    if X_struct is not None and X_struct.shape[1] > 0:
        X_all = np.hstack([X_embed, X_struct])
    else:
        X_all = X_embed

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    acc_scores = []
    prec_scores = []
    rec_scores = []
    f1_scores = []

    fold_num = 1
    for train_idx, test_idx in skf.split(X_all, y):
        X_train, X_test = X_all[train_idx], X_all[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        acc_scores.append(acc)
        prec_scores.append(prec)
        rec_scores.append(rec)
        f1_scores.append(f1)

        print(
            f"  Fold {fold_num}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}"
        )
        fold_num += 1

    print(f"Samples: {len(y)} | Features: {X_all.shape[1]}")
    print(f"Accuracy : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Precision: {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
    print(f"Recall   : {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")
    print(f"F1-score : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")


# Load data
df_model = pd.read_csv(PROCESSED_CSV)
y = prepare_binary_target(df_model["is_a_buyer"])

# Optional structured matrix
X_structured = None
if USE_STRUCTURED_FEATURES:
    X_structured = build_structured_matrix(df_model, STRUCTURED_FEATURES)
    print(f"Using structured features: {STRUCTURED_FEATURES}")
    print(f"Structured matrix shape: {X_structured.shape}")
else:
    print("Using embeddings only (structured features disabled).")

# Evaluate both embedding variants
X_unweighted = load_dense_vectors(UNWEIGHTED_VECTOR)
X_weighted = load_dense_vectors(WEIGHTED_VECTOR)

if len(y) != len(X_unweighted) or len(y) != len(X_weighted):
    raise ValueError("Row count mismatch among labels and embedding vectors.")

Using structured features: ['price', 'product_rating_count', 'brand_encoded', 'product_encoded']
Structured matrix shape: (61284, 4)


In [7]:
evaluate_rf_variant_5fold(
    "Random Forest with Unweighted FastText",
    X_unweighted,
    y,
    X_structured,
)

evaluate_rf_variant_5fold(
    "Random Forest with Weighted FastText",
    X_weighted,
    y,
    X_structured,
)

  Fold 1: Acc=0.8020, Prec=0.8193, Rec=0.9601, F1=0.8841
  Fold 2: Acc=0.7962, Prec=0.8177, Rec=0.9537, F1=0.8804
  Fold 3: Acc=0.8042, Prec=0.8235, Rec=0.9560, F1=0.8848
  Fold 4: Acc=0.8013, Prec=0.8211, Rec=0.9557, F1=0.8833
  Fold 5: Acc=0.8014, Prec=0.8198, Rec=0.9583, F1=0.8836

Random Forest with Unweighted FastText (5-Fold Stratified Cross-Validation Results)
Samples: 61284 | Features: 304
Accuracy : 0.8010 ± 0.0026
Precision: 0.8203 ± 0.0020
Recall   : 0.9568 ± 0.0022
F1-score : 0.8833 ± 0.0015
  Fold 1: Acc=0.8019, Prec=0.8203, Rec=0.9582, F1=0.8839
  Fold 2: Acc=0.7970, Prec=0.8191, Rec=0.9524, F1=0.8807
  Fold 3: Acc=0.8040, Prec=0.8234, Rec=0.9560, F1=0.8848
  Fold 4: Acc=0.8016, Prec=0.8208, Rec=0.9568, F1=0.8836
  Fold 5: Acc=0.7986, Prec=0.8194, Rec=0.9545, F1=0.8818

Random Forest with Weighted FastText (5-Fold Stratified Cross-Validation Results)
Samples: 61284 | Features: 304
Accuracy : 0.8006 ± 0.0025
Precision: 0.8206 ± 0.0015
Recall   : 0.9556 ± 0.0020
F1-score : 